# 0. 라이브러리 인스톨

In [4]:
!pip install lomo-optim optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 28.3 MB/s eta 0:00:00


# 1. 사용 라이브러리 임포트

In [1]:
import torch    # 딥러닝 학습을 위한 torch
import json     # 데이터를 불러올 json
import os
torch.manual_seed(123)  # 토치의 시드를 설정하여 같은 값으로 디버깅

In [2]:
# device GPU(cuda) 설정
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [3]:
# 데이터를 불러올 구글 드라이브 임포트
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# json 데이터 확인 셀
import json

file_path = r'/content/drive/MyDrive/Colab Notebooks/Model_Train/data_set/training_data_10000.jsonl'

try:
    with open(file_path, "r", encoding="utf-8") as f:
        print(f"--- '{file_path}' 파일 내용 ---")
        for i, line in enumerate(f):
          if i >= 10:
            break
            # i+1: 줄 번호 (1부터 시작), line.strip(): 각 줄의 내용 (앞뒤 공백 제거)
          print(f"Line {i+1}: {line.strip()}")
        print("--- 파일 끝 ---")

except FileNotFoundError:
    print(f"오류: '{file_path}' 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
except Exception as e:
    print(f"파일을 읽는 중 오류가 발생했습니다: {e}")

--- '/content/drive/MyDrive/Colab Notebooks/Model_Train/data_set/training_data_10000.jsonl' 파일 내용 ---
Line 1: {"question": "새로 건조된 선박에는 최신 LENGTH OF WATER LINE와(과) middlength beam 시스템이 장착되어 있습니다.", "answers": [{"term": "LENGTH OF WATER LINE", "definition": "수선장"}, {"term": "middlength beam", "definition": "중앙부폭: 계획만재흘수선에서의 중앙부 단면의 폭"}]}
Line 2: {"question": "선박 설계 시 진주채취선의 기준을 반드시 준수해야 합니다.", "answers": [{"term": "진주채취선", "definition": "Pearl Boat: 진주조개를 채취할 목적으로 만든 배"}]}
Line 3: {"question": "매뉴얼에 CATHODIC PROTECTION와(과) 수용시설 관련 내용이 어디에 있는지 찾아봐 주세요.", "answers": [{"term": "CATHODIC PROTECTION", "definition": "음극 방식(陰極防蝕)"}, {"term": "수용시설", "definition": "Reception facilities: 선박으로부터 배출되는 선박평형수 또는 침전물을 수용하는 시설을 말하며, 선박평형수 또는 침전물의 저장, 육상양육 또는 수용시설에서 직접 처리 후 해양으로 배출할 수 있는 시설임."}]}
Line 4: {"question": "이번 프로젝트에서는 WIRE GUY 개념을 이해하는 것이 중요합니다.", "answers": [{"term": "WIRE GUY", "definition": "와이어가이: 와이어로프로 만든 가이"}]}
Line 5: {"question": "OIL BRAKE의 정확한 의미를 아는 사람 있나요?", "answers": [{"term":

# 2. 사용 모델 불러오기 - Qwen3 1.7B

In [4]:
# 허깅페이스 공식 모델 불러오는 법
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

# Qwen3 1.7B 모델 불러오기
model_name = "Qwen/Qwen3-1.7B"

# 모델의 가중치를 32비트가 아닌 16비트로 불러오기 -> GPU 메모리 사용량을 줄이고, 계산 속도도 빨라짐
model_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


# 토크나이저, 모델 불러오기
# 왼쪽 정렬을 하면 마지막 단어가 <pad>로 의미가 없음
# 그래서 오른쪽 정렬을 하면 항상 마지막 단어가 유의미해짐
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left")  # 토크나이저 설정
# 모델 설정
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=model_dtype
    )

model.to(device)        # 모델 GPU 올리기

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/622M [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (up_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (down_proj): Linear(in_features=6144, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2048,), eps=1e-06)
        (post_attention_layer

# 3. 데이터 불러오기

In [8]:
# 허깅페이스 개선 방법
from datasets import load_dataset

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

data_path = "/content/drive/MyDrive/Colab Notebooks/Model_Train/data_set/train_data.jsonl"

# 데이터 셋 불러오기
dataset = load_dataset("json", data_files=data_path, split="train")

Generating train split: 0 examples [00:00, ? examples/s]

In [5]:
from datasets import load_dataset

# 1. 각 파일의 경로를 변수로 지정합니다.
train_file_path = "/content/drive/MyDrive/Colab Notebooks/Model_Train/data_set/train_data.jsonl"
test_file_path = "/content/drive/MyDrive/Colab Notebooks/Model_Train/data_set/test_data.jsonl"

# 2. data_files 인자에 딕셔너리 형태로 전달합니다.
#    'key'가 split의 이름이 되고, 'value'가 해당 파일의 경로가 됩니다.
data_files = {
    "train": train_file_path,
    "test": test_file_path
}

# 3. load_dataset 함수를 호출합니다.
#    이때 split 인자는 생략합니다.
all_datasets = load_dataset("json", data_files=data_files)

# --- 결과 확인 ---
print("전체 데이터셋 정보:")
print(all_datasets)

# 4. 각 데이터셋에 접근할 수 있습니다.
train_dataset = all_datasets["train"]
test_dataset = all_datasets["test"]

print(f"\n학습 데이터셋 크기: {len(train_dataset)}")
print(f"테스트 데이터셋 크기: {len(test_dataset)}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

전체 데이터셋 정보:
DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 16865
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1447
    })
})

학습 데이터셋 크기: 16865
테스트 데이터셋 크기: 1447


In [6]:
# --- 전처리 함수 정의 ---
# 전체 대화 내용을 모델이 학습할 수 있는 단일 텍스트 시퀀스로 변환하는 함수
def formatting_prompts_func(examples):
    # 주어진 대화 내역을 질문과 답변으로 분리
    questions = examples["question"]
    answers = examples["answer"]
    texts = []

    for question, answer in zip(questions, answers):

        # Qwen3의 공식 채팅 템플릿을 사용하여 전체 대화 텍스트 생성
        messages = [
            {"role": "system", "content": "You are an assistant that explains terms about ship building."},
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer}
        ]

        # add_generation_prompt=False: 학습 데이터에는 답변 생성 유도 프롬프트가 필요 없음
        # enable_thinking: 더 깊게 생각하기 모드, 기본설정 True
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False, enable_thinking=True)
        texts.append(text)

    return { "text": texts }    # 딕셔너리 형태로 return

In [9]:
processed_dataset = train_dataset.map(formatting_prompts_func, batched=True, remove_columns=dataset.column_names)
print("\n전처리된 데이터 예시:\n", processed_dataset[1]['text'])

Map:   0%|          | 0/16865 [00:00<?, ? examples/s]


전처리된 데이터 예시:
 <|im_start|>system
You are an assistant that explains terms about ship building.<|im_end|>
<|im_start|>user
According to the report, a problem occurred in the DECIBEL section.<|im_end|>
<|im_start|>assistant
<think>

</think>

보고서에 따르면, 데시벨(소음 측정단위) 부분에서 문제가 발생했습니다.<|im_end|>



In [10]:
test_data = test_dataset.map(formatting_prompts_func, batched=True, remove_columns=dataset.column_names)
print("\n전처리된 데이터 예시:\n", test_data[1]['text'])

Map:   0%|          | 0/1447 [00:00<?, ? examples/s]


전처리된 데이터 예시:
 <|im_start|>system
You are an assistant that explains terms about ship building.<|im_end|>
<|im_start|>user
Tighten the penetration assembly according to the Gas Brazing specification.<|im_end|>
<|im_start|>assistant
<think>

</think>

관통부 시공은 가스 경납땜 규격에 맞춰 체결해 주세요.<|im_end|>



In [11]:
# --- ✨ 2. 누락된 토큰화 단계 추가 ✨ ---
def tokenize_function(examples):
    # 'text' 필드를 토큰화하여 'input_ids'와 'attention_mask'를 생성합니다.
    return tokenizer(
        examples["text"],
        truncation=True,      # max_length보다 길면 자르기
        max_length=2048,      # 모델이 처리할 최대 길이
    )

# 포맷팅된 데이터셋 전체에 토큰화 함수를 적용합니다.
# 이제 데이터셋에는 'input_ids'와 'attention_mask' 필드가 포함됩니다.
tokenized_train_dataset = processed_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"] # 더 이상 필요 없는 'text' 필드 제거
)

tokenized_test_dataset = test_data.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"] # 더 이상 필요 없는 'text' 필드 제거
)

Map:   0%|          | 0/16865 [00:00<?, ? examples/s]

Map:   0%|          | 0/1447 [00:00<?, ? examples/s]

# 4. Train, Test set 분리

In [12]:
from torch.utils.data import random_split
from transformers import DataCollatorForLanguageModeling

total_size = len(processed_dataset)
train_size = 5000
test_size = 1000
unused_size = total_size - train_size - test_size
train_dataset, test_dataset, _ = random_split(tokenized_train_dataset, [train_size, test_size, unused_size])

# 테스트 데이터는 테스트 데이터로 덮어 씌우기
test_dataset = tokenized_test_dataset

print(f"전체 데이터셋 크기: {total_size}")
print(f"학습 데이터셋 크기: {len(train_dataset)}")
print(f"테스트 데이터셋 크기: {len(test_dataset)}")

전체 데이터셋 크기: 16865
학습 데이터셋 크기: 5000
테스트 데이터셋 크기: 1447


In [13]:
from torch.utils.data import DataLoader

# 허깅페이스의 데이터 콜렉터 객체를 생성해서 더욱 편하게 나눠줌 (자동 마스킹, 패딩, 저장 등)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# train, test 로더 설정
train_loader = DataLoader(
    train_dataset,
    batch_size=4, # 예시 배치 사이즈
    shuffle=True,
    collate_fn=data_collator
)

test_loader = DataLoader(
    test_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=data_collator
)

# ★ Optuna 하이퍼파라미터 조정

In [14]:
import torch
import optuna
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, get_scheduler
from tqdm.auto import tqdm
from optuna.exceptions import TrialPruned

# LOMO 옵티마이저 import
from lomo_optim import Lomo
from lomo_optim import AdaLomo

In [16]:
# 2. Objective 함수 정의 (LOMO Full Fine-tuning 용)
def objective(trial):
    # =========================================
    # 파라미터 범주 설정
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True)
    num_train_epochs = trial.suggest_int("num_train_epochs", 1, 3)

    # --- 메모리 관리를 위한 하이퍼파라미터 고정 ---
    batch_size = trial.suggest_categorical("batch_size", [2, 4, 8])

    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.05)
    warmup_ratio = trial.suggest_float("warmup_ratio", 0.0, 0.1)
    lr_scheduler_type = trial.suggest_categorical("lr_scheduler_type", ["linear", "cosine", "cosine_with_restarts"])

    # ===========================================
    # 매 시도 마다 모델, 데이터 로더, 옵티마이저 등 새로 초기화
    # 모델 설정
    model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=model_dtype)
    model.resize_token_embeddings(len(tokenizer))
    model.gradient_checkpointing_enable()

    model.to(device)

    # 2. 데이터 로더 생성
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=data_collator)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, collate_fn=data_collator)

    # 3. ✨ LOMO 옵티마이저 설정 ✨
    optimizer = Lomo(model, lr=learning_rate, weight_decay=weight_decay)

    # 4. 학습률 스케줄러 설정
    num_training_steps = num_train_epochs * len(train_loader)
    num_warmup_steps = int(num_training_steps * warmup_ratio)
    lr_scheduler = get_scheduler(
        name=lr_scheduler_type,
        optimizer=optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps
    )

    # ===========================================
    # 학습 및 평가 루프
    for epoch in range(num_train_epochs):
        model.train()
        progress_bar = tqdm(train_loader, desc=f"Trial {trial.number} Epoch {epoch+1}/{num_train_epochs}", leave=False)

        for batch in progress_bar:
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)
            loss = outputs.loss

            # LOMO는 backward()에서 파라미터 업데이트까지 처리
            loss.backward()

            optimizer.zero_grad()
            lr_scheduler.step()
            progress_bar.set_postfix(loss=loss.item())

        # --- 프루닝(가지치기) 로직 ---
        model.eval()
        total_eval_loss = 0
        with torch.no_grad():
            for batch in test_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                outputs = model(**batch)
                loss = outputs.loss
                total_eval_loss += loss.item()

        avg_eval_loss = total_eval_loss / len(test_loader)
        trial.report(avg_eval_loss, epoch)

        if trial.should_prune():
            raise TrialPruned()

    return avg_eval_loss

In [ ]:
# -------------------------------------------------------------------
# ## 3. Study 객체 생성 및 최적화 실행
# -------------------------------------------------------------------
from optuna.pruners import MedianPruner


study = optuna.create_study(direction="minimize", pruner=MedianPruner())
# 50번 50번 나눠서 진행
study.optimize(objective, n_trials=50)

best_params = study.best_params

# --- 결과 확인 ---
print("="*50)
print("최적화 종료!")
print("최고 점수 (loss):", study.best_trial.value)
print("최적 하이퍼파라미터:", study.best_params)
print("="*50)

[I 2025-10-22 00:22:26,440] A new study created in memory with name: no-name-559a1d3c-8057-475f-b83b-7fb4e4c0b805


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Trial 0 Epoch 1/2:   0%|          | 0/1250 [00:00<?, ?it/s]

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Trial 0 Epoch 2/2:   0%|          | 0/1250 [00:00<?, ?it/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Trial 42 Epoch 1/1:   0%|          | 0/625 [00:00<?, ?it/s]

[I 2025-10-22 08:04:35,581] Trial 42 pruned. 


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Trial 43 Epoch 1/1:   0%|          | 0/625 [00:00<?, ?it/s]

In [1]:
print('50번 최적화 진행 후 최적의 파라미터')
print(best_params)

50번 최적화 진행 후 최적의 파라미터


NameError: name 'best_params' is not defined

In [2]:
# 50번 50번 나눠서 진행
study.optimize(objective, n_trials=50)

best_params = study.best_params

# --- 결과 확인 ---
print("="*50)
print("최적화 종료!")
print("최고 점수 (loss):", study.best_trial.value)
print("최적 하이퍼파라미터:", study.best_params)
print("="*50)

NameError: name 'study' is not defined

In [ ]:
print('100번 최적화 진행 후 최적의 파라미터')
print(best_params)

# 5. LOMO로 학습 진행

In [ ]:
from tqdm.auto import tqdm
from transformers import get_scheduler
from lomo_optim import AdaLomo


# --- 하이퍼 파라미터 설정 ---
learning_rate = 1e-4
num_epochs = 2
# -----------------------------

# LOMO는 AdamW와 같은 옵티마이저 상태를 저장하지 않아 메모리를 절약합니다.
optimizer = AdaLomo(model, lr=learning_rate)

# 학습률 스케줄러 설정
num_training_steps = num_epochs * len(train_loader)

lr_scheduler = get_scheduler(
    name="linear",
    optimizer=optimizer,
    num_warmup_steps=0, # LOMO 사용 시에는 웜업을 사용하지 않는 경우가 많습니다.
    num_training_steps=num_training_steps
)

In [ ]:
for epoch in range(num_epochs):
    model.train()
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}")

    for batch in progress_bar:
        # DataCollator가 반환한 배치를 GPU로 이동
        batch = {k: v.to(device) for k, v in batch.items()}

        # 1. Forward Pass: 모델을 통해 예측(logits)과 손실(loss)을 계산
        outputs = model(**batch)
        loss = outputs.loss

        # 2. Backward Pass + Parameter Update (LOMO의 핵심)
        # LOMO는 backward() 호출 시 내부적으로 파라미터 업데이트까지 수행합니다.
        loss.backward()

        # 3. 그래디언트 초기화 및 스케줄러 스텝
        # backward() 후에 그래디언트를 초기화합니다.
        optimizer.zero_grad()
        lr_scheduler.step()

        progress_bar.set_postfix(loss=loss.item())

Epoch 1/2:   0%|          | 0/1250 [00:00<?, ?it/s]

Epoch 2/2:   0%|          | 0/1250 [00:00<?, ?it/s]

# 6. 성능 평가

In [ ]:
import math

model.eval()
total_eval_loss = 0

# 평가 시에는 가중치를 업데이트하지 않으므로, 불필요한 계산을 막아 메모리를 절약하고 속도를 높입니다.
with torch.no_grad():
    # test_loader를 사용하여 평가 데이터에 대한 루프 실행
    for batch in tqdm(test_loader, desc="Evaluating"):
        batch = {k: v.to(device) for k, v in batch.items()}

        # Forward Pass 실행
        outputs = model(**batch)
        loss = outputs.loss

        # 각 배치의 loss를 누적
        total_eval_loss += loss.item()

# 3. 평균 평가 손실(Average Evaluation Loss) 계산
avg_eval_loss = total_eval_loss / len(test_loader)

# 4. 퍼플렉시티(Perplexity) 계산
#    Perplexity는 e^(loss) 입니다. 값이 낮을수록 모델이 다음 단어를 잘 예측한다는 의미입니다.
try:
    perplexity = math.exp(avg_eval_loss)
except OverflowError:
    perplexity = float("inf") # loss가 너무 클 경우 무한대로 표시

# --- 5. 평가 결과 출력 ---
print("\n--- 평가 결과 ---")
print(f"평균 평가 손실 (Average Eval Loss): {avg_eval_loss:.4f}")
print(f"퍼플렉시티 (Perplexity): {perplexity:.4f}")
print("="*20)

Evaluating:   0%|          | 0/362 [00:00<?, ?it/s]


--- 평가 결과 ---
평균 평가 손실 (Average Eval Loss): 2.4700
퍼플렉시티 (Perplexity): 11.8227


# 7. 모델 저장

In [ ]:
# --- 최종 모델 저장 (Hugging Face 형식) ---

# 1. 저장할 '폴더'의 경로를 지정합니다. (파일 이름이 아님)
output_dir = "/content/drive/MyDrive/main/Model_Train/best_model/Qwen3_(1dot7B)_LOMO"

# 2. .save_pretrained() 메서드를 사용하여 모델과 토크나이저를 저장합니다.
#    이 메서드가 알아서 safetensors와 config.json 등을 생성합니다.
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"\n✅ 최종 모델이 Hugging Face 형식으로 '{output_dir}' 폴더에 저장되었습니다.")

# (확인) 저장된 파일 목록을 출력해 봅니다.
# print("\n--- 저장된 파일 목록 ---")
# !ls -l {output_dir}


✅ 최종 모델이 Hugging Face 형식으로 '/content/drive/MyDrive/main/Model_Train/best_model/Qwen3_(1dot7B)_LOMO' 폴더에 저장되었습니다.


In [ ]:
from transformers import TextStreamer

# 모델을 평가 모드로 설정
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"

def generate_answer(question):
    """
    올바른 채팅 템플릿을 사용하여 답변을 생성하는 함수.
    """
    # 1. 시스템 메시지와 사용자 질문으로 대화 형식 구성
    messages = [
        {"role": "system", "content": "You are an assistant that explains terms about ship building."},
        {"role": "user", "content": question}
    ]

    # 2. tokenizer.apply_chat_template을 사용하여 Qwen3의 공식 프롬프트 형식으로 변환
    #    add_generation_prompt=True가 모델에게 답변을 시작하라는 신호를 줍니다.
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # 3. 프롬프트를 토큰화하여 모델 입력으로 변환
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # 4. 모델을 통해 답변 생성
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=512)

    # 5. 생성된 결과에서 입력 프롬프트 부분을 제외하고 디코딩
    answer = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)

    return answer

# --- 테스트 ---
while True:
    my_question = input()

    if my_question == '탈출':
        break

    final_answer = generate_answer(my_question)
    print(final_answer)

Draft의 뜻이 뭐야
<think>

</think>

Draft의 의미는 무엇인가요?
What is mean about draft?
<think>

</think>

해당 항목은 선박 설계 시 수중 중심 위치의 높이를 나타냅니다.


KeyboardInterrupt: Interrupted by user

# ★★ Optuna로 최적화 한 하이퍼파라미터로 다시 학습

In [ ]:
# 최종 학습을 위한 모델 불러오기
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=model_dtype
    )

model.resize_token_embeddings(len(tokenizer)) # 어휘 크기 동기화
model.gradient_checkpointing_enable()         # 메모리 최적화
model.to(device)

train_loader = DataLoader(train_dataset, batch_size=best_params['batch_size'], shuffle=True, collate_fn=data_collator)
test_loader = DataLoader(test_dataset, batch_size=best_params['batch_size'], collate_fn=data_collator)

optimizer = Lomo(model, lr=best_params['learning_rate'])

num_epochs = best_params['num_train_epochs']
num_training_steps = num_epochs * len(train_loader)
num_warmup_steps = int(num_training_steps * best_params['warmup_ratio'])

lr_scheduler = get_scheduler(
    name=best_params['lr_scheduler_type'],
    optimizer=optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

In [ ]:
print("\n--- 최종 모델 학습 시작 ---")
for epoch in range(num_epochs):
    model.train()
    progress_bar = tqdm(train_loader, desc=f"Final Training Epoch {epoch+1}/{num_epochs}")

    for batch in progress_bar:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.zero_grad()
        lr_scheduler.step()
        progress_bar.set_postfix(loss=loss.item())

print("✅ 최종 학습 완료!")

In [ ]:
import math
# --- 4. 최종 성능 평가 ---
print("\n--- 최종 모델 성능 평가 시작 ---")
model.eval()
total_eval_loss = 0
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Final Evaluation"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        total_eval_loss += loss.item()

avg_eval_loss = total_eval_loss / len(test_loader)
perplexity = math.exp(avg_eval_loss)

print("\n--- 최종 평가 결과 ---")
print(f"평균 평가 손실 (Loss): {avg_eval_loss:.4f}")
print(f"퍼플렉시티 (Perplexity): {perplexity:.4f}")
print("="*25)

In [ ]:
output_path = r'/content/drive/MyDrive/Colab Notebooks/Model_Train/best_model'
save_name=r'/Qwen3_(1dot7B)_LOMO_5000_t100'
output_dir = output_path+save_name

# 2. .save_pretrained() 메서드를 사용하여 모델과 토크나이저를 저장합니다.
#    이 메서드가 알아서 safetensors와 config.json 등을 생성합니다.
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"\n✅ 최종 모델이 Hugging Face 형식으로 '{output_dir}' 폴더에 저장되었습니다.")